# 避難所候補算出支援ツール（sheltermatch）

要支援者一覧（緯度・経度入り）と避難所一覧の座標から、要支援者ごとに **距離が近い避難所候補（既定で上位3件）** を
算出し、必要であればハザード区域との位置関係も確認して、職員の最終判断のための資料（CSV）を作成するNotebookです。

**要支援者一覧CSVの正式フォーマット**: `resident_id,address,latitude,longitude,geocode_status` の
5列です（テンプレート: `templates/residents.csv`）。これは別Notebook `address_geocode.ipynb`
（住所→座標変換）の出力と同じ形式のため、`address_geocode.ipynb` で変換したCSVは、**加工せずそのまま
ここへアップロード**できます。

```text
templates/residents.csv
  ↓
address_geocode.ipynb（住所→座標変換）
  ↓
同じ5列のCSV
  ↓
sheltermatch.ipynb（このNotebook）
```

住所しかない場合は、先に `address_geocode.ipynb` で緯度・経度を付与してください（このNotebook自体は
住所→座標変換は行いません）。5列がすべて揃っていない要支援者CSVは受け付けません。

**このツールが行うこと**
- 直線距離（`geopy.distance.geodesic`）による避難所候補の算出（候補提示。自動割当ではありません）
- 任意機能として、要支援者・避難所・両者を結ぶ直線とハザード区域（GeoJSON / Shapefile。国・県等の
  公式配布ZIPも展開・変換せずそのままアップロード可能）との位置関係の確認

**このツールが行わないこと（重要）**
- 住所→座標変換（要支援者一覧CSVの `latitude` / `longitude` にあらかじめ座標を入力してください。
  住所しかない場合は、別Notebookの `address_geocode.ipynb` で事前に緯度・経度を付与してから、
  その出力CSVをここへアップロードしてください）
- 避難先・避難経路の自動決定、道路経路・通行可能性の計算（直線距離であり道路距離ではありません。
  直線とハザード区域の交差判定も、道路上の避難経路判定ではありません）
- ハザード判定結果による候補避難所の自動除外・自動順位変更、独自の危険度スコアリング

距離順位とハザード判定は別々の情報として出力します。どの避難所を選ぶかは、出力結果を確認した **職員が判断** してください。

**ハザード判定結果の読み方**
- `True` = ハザード区域内（境界上含む）または交差あり
- `False` = 有効な座標で判定した結果、ハザード区域外
- 空欄（NaN） = 座標が無い・不正、または候補自体が無いなどの理由で **判定できなかった**（区域外という意味ではありません）

**個人情報の取り扱い**
- 実際の要支援者データ・ハザードデータはこのリポジトリにコミットしないでください（サンプルを追加する場合は完全な架空データを使用してください）。
- 出力CSVには住所・座標等の個人情報が含まれ得ます。取り扱いに注意してください。

## 使い方

1. 下の「利用者設定」を確認する
2. 「ランタイム → すべてのセルを実行」
3. 要支援者CSV（`resident_id,address,latitude,longitude,geocode_status` の5列必須。
   `latitude` / `longitude` の値は行ごとに欠損・不正でも構いません。`resident_id` は空欄・重複不可です。
   `address_geocode.ipynb` の出力CSVはそのままアップロードできます）をアップロードする
4. 必要な場合のみハザードデータ（GeoJSON・Shapefile、または国・県等の公式配布ZIP）をアップロードする
5. BODIK Data APIからの避難所取得に失敗した場合のみ、避難所CSVをアップロードする
6. 結果CSV（`assigned_shelters.csv`）を保存する

コードを読み込まなくても、この説明と各セルのprint出力だけで操作できます。


In [ ]:
# ===== 利用者設定 =====
# 通常変更が必要な項目はこれだけです。値を確認・変更してから実行してください。

# 要支援者・避難所とハザード区域（GeoJSON）との位置関係を確認する場合は True にしてください。
ENABLE_HAZARD_CHECK = False

# 要支援者ごとに算出する避難所候補の件数（避難所がこの件数未満の場合は存在する件数まで出力）
TOP_N = 3

# 避難所一覧の取得方法。"api"=BODIK Data APIから取得 / "csv"=CSVファイルをアップロード
# "api"で取得に失敗した場合は、自動的にCSVアップロードへ切り替わります。
SHELTER_SOURCE = "api"

if not isinstance(TOP_N, int) or TOP_N < 1:
    raise ValueError(f"TOP_N は1以上の整数を指定してください。現在の値: {TOP_N!r}")

if SHELTER_SOURCE not in ("api", "csv"):
    raise ValueError(f"SHELTER_SOURCE は 'api' または 'csv' を指定してください。現在の値: {SHELTER_SOURCE!r}")

print("利用者設定を読み込みました。")
print(f"  ENABLE_HAZARD_CHECK = {ENABLE_HAZARD_CHECK}")
print(f"  TOP_N               = {TOP_N}")
print(f"  SHELTER_SOURCE      = '{SHELTER_SOURCE}'")


In [ ]:
# ===== 実行環境準備 =====
# Google Colabに標準で入っていないライブラリをインストールします（初回のみ数十秒かかることがあります）。
%pip install -q geopy geopandas shapely

import io
import re
import tempfile
import time
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import requests
from geopy.distance import geodesic

import geopandas as gpd
from shapely.geometry import Point, LineString

from google.colab import files

# 工程別の処理時間（秒）を記録する軽量な計測用の入れ物。time.perf_counter()による簡易計測のためだけに
# 追加したもので、既存の処理結果・ロジックは変更しない（詳細なprofilingは行わない）。
TIMINGS = {}

print("ライブラリの読み込みが完了しました。")


In [ ]:
# ===== 要支援者CSV読込・入力チェック =====
# 要支援者一覧CSVを選択してください（ファイル名は自由です）。共通住民CSVの正式フォーマットは
# resident_id,address,latitude,longitude,geocode_status の5列です（address_geocode.ipynbの
# 出力と同じ形式のため、そのままアップロードできます）。5列すべてを必須列として確認し、
# 不足していれば処理を止めます。resident_idは住所変換結果との対応が崩れないよう、空欄・重複が
# あればここでも処理を止めます（address_geocode.ipynbと同じ最低限のチェック）。
# latitude / longitude の値は行ごとに欠損・不正でも構いません（行を削除せず、後続の距離計算のみ
# 対象外とします。match_status 列で no_coordinates / invalid_coordinates として確認できます）。
# geocode_status列は変更せずそのまま結果CSVへ保持します（座標→避難所候補の突合結果である
# sheltermatch自身のmatch_status列とは別の情報のため、混同しないでください）。
# 文字コードは UTF-8 (BOM付き) → CP932 → UTF-8 の順に自動判定します。resident_idは
# address_geocode.ipynbと同様、先頭ゼロ等の表記を保つため文字列として読み込みます。

print("【要支援者一覧CSV】を選択してください。")
uploaded_residents = files.upload()

if len(uploaded_residents) == 0:
    raise RuntimeError("要支援者一覧CSVがアップロードされませんでした。ファイルを1つ選択してください。")
if len(uploaded_residents) > 1:
    raise RuntimeError("要支援者一覧CSVは1つだけ選択してください。")

residents_filename = list(uploaded_residents.keys())[0]
residents_bytes = uploaded_residents[residents_filename]
print(f"'{residents_filename}' を要支援者一覧として受け取りました。")

# 要支援者CSVを新たに読み込むたびに、この時点から性能計測をやり直す（TIMINGSをリセットする）。
# ハザードデータ・避難所データを読み込み直さずに、この「要支援者CSV読込・入力チェック」セルから
# 100/500/1000件を切り替えて再実行した場合に、前回実行分のhazard/shelters等の時間が今回の
# 合計へ混入しないようにするための措置（ハザードGeoDataFrame・避難所DataFrame自体は再利用できる。
# 計測用の辞書だけを空にする）。
TIMINGS = {}

_stage_start = time.perf_counter()


def read_csv_auto(file_bytes, label, dtype=None):
    """UTF-8(BOM付き) → CP932 → UTF-8 の順で読み込みを試み、成功したDataFrameを返す。"""
    encodings = [
        ("utf-8-sig", "UTF-8 (BOM付き)"),
        ("cp932", "CP932 (Shift-JIS系)"),
        ("utf-8", "UTF-8"),
    ]
    last_error = None
    for encoding, encoding_label in encodings:
        try:
            df = pd.read_csv(io.BytesIO(file_bytes), encoding=encoding, dtype=dtype)
            print(f"[{label}] {encoding_label} として読み込みました。（{len(df)}行）")
            return df
        except (UnicodeDecodeError, UnicodeError) as error:
            last_error = error
            continue
    raise ValueError(
        f"[{label}] 文字コードを判定できませんでした。"
        "UTF-8(BOM付き)・CP932・UTF-8のいずれでも読み込めません。"
        "Excel等での保存時の文字コードを確認してください。"
        f" 詳細: {last_error}"
    )


def ensure_coordinate_columns(df, label):
    """latitude/longitude列が両方存在することを確認する（値の欠損・不正は許容し、ここでは
    行を削除しない。列自体が無い場合のみ列不足として停止する）。避難所一覧CSV用。"""
    missing = [c for c in ("latitude", "longitude") if c not in df.columns]
    if missing:
        raise ValueError(f"[{label}] 必須列が見つかりません: {', '.join(missing)}")
    return df


# 共通住民CSVの正式フォーマット（address_geocode.ipynbの入出力と同じ5列）。
RESIDENT_REQUIRED_COLUMNS = ["resident_id", "address", "latitude", "longitude", "geocode_status"]


def ensure_resident_columns(df, label):
    """要支援者一覧CSVが共通住民CSVの5列（resident_id,address,latitude,longitude,geocode_status）
    をすべて持つことを確認する。列そのものは5列とも必須とし、不足していれば処理を止める
    （住所変換input＝output＝sheltermatch inputという共通フォーマットを一本化するため、
    旧来のlatitude/longitudeのみの形式は正式入力として扱わない）。latitude/longitudeの
    セルの値が欠損・不正な行は、従来どおり削除せず許容する。"""
    missing = [c for c in RESIDENT_REQUIRED_COLUMNS if c not in df.columns]
    if missing:
        raise ValueError(
            f"[{label}] 必須列が見つかりません: {', '.join(missing)}\n"
            f"共通住民CSVは {','.join(RESIDENT_REQUIRED_COLUMNS)} の5列です。"
            "templates/residents.csv を使用してください。"
        )
    return df


def find_resident_id_problems(df):
    """resident_id列の空欄・重複を検出する。resident_idは住所変換結果と避難所候補算出結果を
    紐づける結合キーのため、曖昧な状態のまま処理を進めない（address_geocode.ipynbと同じチェック）。
    戻り値は問題を説明する文字列のリスト（問題が無ければ空リスト）。行番号はCSV上の行番号
    （1行目をヘッダーとする）で示す。"""
    problems = []
    resident_id = df["resident_id"]

    blank_mask = resident_id.isna() | (resident_id.astype(str).str.strip() == "")
    if blank_mask.any():
        blank_rows = [i + 2 for i in df.index[blank_mask]]
        problems.append(f"resident_id が空欄の行があります（CSV行番号: {', '.join(map(str, blank_rows))}）")

    dup_mask = resident_id.duplicated(keep=False) & ~blank_mask
    if dup_mask.any():
        dup_detail = [f"CSV行{i + 2}='{resident_id.iloc[i]}'" for i in df.index[dup_mask]]
        problems.append(f"resident_id が重複している行があります（{', '.join(dup_detail)}）")

    return problems


def is_valid_coordinate(lat, lon):
    """緯度・経度が数値として有効な範囲かどうかを返す（欠損・範囲外はFalse）。
    距離計算・ハザード判定など、1点ずつ座標を扱う関数から共通で利用する。"""
    if pd.isna(lat) or pd.isna(lon):
        return False
    return (-90 <= lat <= 90) and (-180 <= lon <= 180)


def parse_and_validate_coordinates(df, label):
    """latitude/longitude列を数値化し、有効な座標かどうかの真偽値Seriesを返す。"""
    lat = pd.to_numeric(df["latitude"], errors="coerce")
    lon = pd.to_numeric(df["longitude"], errors="coerce")
    valid = lat.notna() & lon.notna() & lat.between(-90, 90) & lon.between(-180, 180)
    invalid_count = int((~valid).sum())
    if invalid_count:
        print(f"[{label}] 座標が欠損・不正な行が {invalid_count}件あります（全{len(df)}行中）。")
    return lat, lon, valid


residents_raw = read_csv_auto(residents_bytes, "要支援者一覧", dtype={"resident_id": str})
residents_raw = ensure_resident_columns(residents_raw, "要支援者一覧")

resident_id_problems = find_resident_id_problems(residents_raw)
if resident_id_problems:
    raise ValueError(
        "[要支援者一覧] resident_id の内容に問題があるため処理を中止しました。\n"
        + "\n".join(f"  - {problem}" for problem in resident_id_problems)
        + "\nresident_id を空欄・重複のない一意な値に修正してから、再度アップロードしてください。"
    )

residents = residents_raw.copy()
residents["latitude"], residents["longitude"], residents_coord_valid = parse_and_validate_coordinates(
    residents, "要支援者一覧"
)

display(residents_raw.head())

TIMINGS["residents_csv"] = time.perf_counter() - _stage_start
print(f"[処理時間] 要支援者CSV読込・入力チェック: {TIMINGS['residents_csv']:.2f}秒")


In [ ]:
# ===== 避難所取得・正規化・入力チェック =====
# 避難所一覧は上の「利用者設定」の SHELTER_SOURCE に従って取得します。
# - "api"（既定）: BODIK Data API（CKANの datastore_search）から自治体標準ODSの避難所データを
#   直接取得します（公開データの読み取りのみのためAPIキーは不要）。取得に失敗した場合
#   （通信エラー・レスポンス異常・0件など）は、Notebookを停止せずCSVアップロードへ自動的に切り替えます。
# - "csv": 避難所一覧CSVをブラウザから選択してアップロードします。
#
# 避難所一覧CSVが自治体標準オープンデータセット(ODS)形式（例:
# https://data.bodik.jp/dataset/472107_evacuation_space ）の場合、日本語列名
# （名称→name、緯度→latitude、経度→longitude）を自動的に内部標準列名へ変換します。
# 従来の name/latitude/longitude 形式のCSVもそのまま利用できます。
# 「災害種別_」で始まる列がある場合は、値の 1(対応済み)/2(2階以上であれば対応済み)/空欄(未対応) の
# 区別を保ったまま、避難所候補ごとにCSVへ参考情報として出力します（距離順位や候補の自動除外には
# 使用しません。これは指定緊急避難場所としての対応種別情報であり、後述のGeoJSONによるハザード判定
# とは別の情報です）。
# 座標が不正な避難所の行は距離計算の対象から除外します。

_stage_start = time.perf_counter()

BODIK_BASE_URL = "https://data.bodik.jp"
BODIK_RESOURCE_ID = "3132a0a4-f522-4b2d-bf18-f106d8b3a5ae"  # 糸満市 指定緊急避難場所データセット


def fetch_shelters_from_bodik(base_url, resource_id, page_size=1000):
    """BODIKのCKAN Data API(datastore_search)から避難所データを全件取得し、DataFrameで返す。
    total/offsetでページングして全件取得する。取得できない場合は例外を発生させ、
    呼び出し側でCSVアップロードへフォールバックする。"""
    endpoint = f"{base_url}/api/action/datastore_search"
    records = []
    offset = 0
    total = None

    while True:
        response = requests.get(
            endpoint,
            params={"resource_id": resource_id, "limit": page_size, "offset": offset},
            timeout=30,
        )
        response.raise_for_status()
        payload = response.json()

        if not payload.get("success"):
            raise RuntimeError("CKAN APIレスポンスが success=false を返しました。")

        result = payload.get("result")
        if result is None or "records" not in result:
            raise RuntimeError("CKAN APIレスポンスに result.records が含まれていません。")

        page_records = result["records"]
        records.extend(page_records)

        if total is None:
            total = result.get("total", len(page_records))

        offset += len(page_records)
        if len(page_records) == 0 or offset >= total:
            break

    if len(records) == 0:
        raise RuntimeError("BODIK APIの取得結果が0件でした。")

    return pd.DataFrame(records)


shelters_raw = None
shelters_bytes = None

if SHELTER_SOURCE == "api":
    try:
        shelters_raw = fetch_shelters_from_bodik(BODIK_BASE_URL, BODIK_RESOURCE_ID)
        print(f"BODIK APIから避難所一覧を{len(shelters_raw)}件取得しました。")
    except Exception as error:
        print("BODIK APIから避難所一覧を取得できませんでした。")
        print("CSVファイルから読み込みます。")
        print(f"（詳細: {error}）")

if shelters_raw is None:
    print("【避難所一覧CSV】を選択してください。")
    uploaded_shelters = files.upload()

    if len(uploaded_shelters) == 0:
        raise RuntimeError("避難所一覧CSVがアップロードされませんでした。ファイルを1つ選択してください。")
    if len(uploaded_shelters) > 1:
        raise RuntimeError("避難所一覧CSVは1つだけ選択してください。")

    shelters_filename = list(uploaded_shelters.keys())[0]
    shelters_bytes = uploaded_shelters[shelters_filename]
    print(f"'{shelters_filename}' を避難所一覧として受け取りました。")

if shelters_bytes is not None:
    shelters_raw = read_csv_auto(shelters_bytes, "避難所一覧")

# 避難所一覧の列名アイリアス（自治体標準ODS等の日本語列名 → 内部標準列名）
SHELTER_COLUMN_ALIASES = {
    "name": ["名称"],
    "latitude": ["緯度"],
    "longitude": ["経度"],
}

# 避難所の災害種別列の接頭辞（この接頭辞で始まる列を動的にすべて認識する）
DISASTER_TYPE_COLUMN_PREFIX = "災害種別_"

# 自治体標準ODSの災害種別列の値と対応区分の対応表（BODIKの指定緊急避難場所データセット仕様）
# 1=対応済み、2=2階以上であれば対応済み、空欄=未対応。これ以外の値は独自に「対応」と推測しない。
DISASTER_SUPPORT_LABELS = {
    "1": "対応済み",
    "1.0": "対応済み",
    "2": "2階以上であれば対応済み",
    "2.0": "2階以上であれば対応済み",
}


def normalize_shelter_columns(df, label):
    """自治体標準ODS等で使われる日本語列名を、内部標準列名(name/latitude/longitude)へ変換する。
    既に標準列名がある場合はそちらを優先し、変換しない。使用した列名の対応表も返す。"""
    df = df.copy()
    used_columns = {}
    for standard_col, aliases in SHELTER_COLUMN_ALIASES.items():
        if standard_col in df.columns:
            used_columns[standard_col] = standard_col
            continue
        for alias in aliases:
            if alias in df.columns:
                df = df.rename(columns={alias: standard_col})
                used_columns[standard_col] = alias
                break

    renamed = {std: orig for std, orig in used_columns.items() if orig != std}
    if renamed:
        mapping_text = ", ".join(f"{orig}→{std}" for std, orig in renamed.items())
        print(f"[{label}] 自治体標準ODS等の日本語列名を自動変換しました: {mapping_text}")

    return df, used_columns


def detect_disaster_type_columns(df):
    """列名が '災害種別_' で始まる列を、対応する災害種別情報として動的に検出する。
    特定の災害種別名の一覧に固定せず、実データに存在する列をそのまま採用する。"""
    return [c for c in df.columns if c.startswith(DISASTER_TYPE_COLUMN_PREFIX)]


def classify_disaster_support(value):
    """自治体標準ODSの災害種別列の値を解釈し、対応区分のラベルを返す（未対応ならNone）。
    '1'=対応済み、'2'=2階以上であれば対応済み、空欄=未対応という標準仕様に従う。
    それ以外の想定外の値は独自に「対応している」と推測せず、値をそのまま保持して要確認として返す。"""
    if pd.isna(value):
        return None
    text = str(value).strip()
    if text == "":
        return None
    if text in DISASTER_SUPPORT_LABELS:
        return DISASTER_SUPPORT_LABELS[text]
    return f"{text}(要確認)"


def build_disaster_support(df, disaster_columns):
    """行ごとに対応している災害種別とその対応区分を ';' 区切りでまとめたSeriesを返す。
    例: '洪水:対応済み;高潮:2階以上であれば対応済み'。未対応(空欄)の種別は含めない。
    災害種別列が無ければ全行NaN。"""
    if not disaster_columns:
        return pd.Series([np.nan] * len(df), index=df.index, dtype="object")

    prefix_len = len(DISASTER_TYPE_COLUMN_PREFIX)

    def row_support(row):
        parts = []
        for col in disaster_columns:
            label = classify_disaster_support(row[col])
            if label is not None:
                parts.append(f"{col[prefix_len:]}:{label}")
        return ";".join(parts)

    return df.apply(row_support, axis=1)


# 避難所一覧は座標列の存否を確認する前に、自治体標準ODS等の日本語列名を内部標準列名へ変換する
shelters_raw, shelter_used_columns = normalize_shelter_columns(shelters_raw, "避難所一覧")
shelter_disaster_columns = detect_disaster_type_columns(shelters_raw)
shelters_raw["_disaster_support"] = build_disaster_support(shelters_raw, shelter_disaster_columns)
HAS_DISASTER_TYPE_COLUMNS = len(shelter_disaster_columns) > 0

shelters_raw = ensure_coordinate_columns(shelters_raw, "避難所一覧")
if "name" not in shelters_raw.columns:
    raise ValueError("[避難所一覧] 名称列が見つかりません: name または 名称 の列が必要です。")

shelters = shelters_raw.copy()
shelters["latitude"], shelters["longitude"], shelters_coord_valid = parse_and_validate_coordinates(
    shelters, "避難所一覧"
)

# 座標が不正な避難所の行は距離計算の対象から除外する（有効な避難所が0件の場合は処理を停止する）
invalid_shelter_count = int((~shelters_coord_valid).sum())
shelters_valid = shelters.loc[shelters_coord_valid].reset_index(drop=True)

if len(shelters_valid) == 0:
    raise RuntimeError(
        "有効な座標を持つ避難所が0件です。避難所一覧CSVの latitude / longitude 列を確認してください。"
    )

print(
    f"[避難所一覧] 避難所件数: {len(shelters_raw)}件 / "
    f"名称列: '{shelter_used_columns.get('name', 'name')}' / "
    f"緯度列: '{shelter_used_columns.get('latitude', 'latitude')}' / "
    f"経度列: '{shelter_used_columns.get('longitude', 'longitude')}' / "
    f"認識した災害種別列数: {len(shelter_disaster_columns)}件"
)
print(f"距離計算に使用する有効な避難所: {len(shelters_valid)}件（座標不正のため{invalid_shelter_count}件を除外）")

display(shelters_raw.head())

TIMINGS["shelters"] = time.perf_counter() - _stage_start
print(f"[処理時間] 避難所データ取得・整形: {TIMINGS['shelters']:.2f}秒")


## ハザードデータ読込（任意）

`ENABLE_HAZARD_CHECK = True`（上の「利用者設定」）の場合のみ、ハザード区域のデータをアップロードします。
1ファイルにつき **`.geojson` 単体**、または **国・県等の公式配布ZIP（`.zip`）** のいずれかを選択できます。
ZIPの場合、内部のディレクトリ構造やファイル数に関わらず、配下の `.geojson` と **Shapefile**（`.shp`
とその組の `.shx`/`.dbf`。`.prj` は無くても構いません）を自動的に再帰探索してすべて統合します
（利用者が展開・変換・整理する必要はありません）。ZIP内のファイル名がCP932（Shift-JIS系）でUTF-8
フラグを立てずに格納されている場合（沖縄県公式データ等でよく見られます）も、自動的に文字化けを復元
してから展開します。`.shx`/`.dbf` が揃っていないShapefileは、そのレイヤーのみ読み込めない旨をまとめて
表示し、ほかに有効なレイヤーがあれば処理を継続します。区域判定に使えるPolygon/MultiPolygon以外の
ジオメトリ（LineString等）が含まれる場合は、独自に面へ変換したりはせず区域判定対象外として除外します。

アップロードした **ファイル（GeoJSONまたはZIP）ごとに1回だけ** ハザード種別名を決定します（ZIP内の個々の
レイヤーごとには入力しません）。ファイル名が以下のような既知の公式配布形式に一致する場合は、入力を求めず
自動判定します（一致しない場合のみ、これまでどおり種別名の入力を求めます。空欄の場合はファイル名を
`hazard_type` とする挙動も維持されます）。

- 国土数値情報「洪水浸水想定区域データ」（`A31a-` で始まる形式。例: `A31a-25_47_10_GEOJSON.zip`）
  → 河川区分から「洪水（洪水予報河川・水位周知河川）」/「洪水（その他の河川）」を判定
- 国土数値情報「土砂災害警戒区域データ」（`A33-` で始まる形式。例: `A33-25_47_GEOJSON.zip`）→「土砂災害」
  （属性 `A33_001`/`A33_002` があれば現象の種類・区域区分も反映）
- 沖縄県津波浸水想定データ（`level1`〜`level7` で始まる形式。例: `level1_1cm-30cm.zip`）→「津波」
- 沖縄県高潮浸水想定データ（`<市町村コード>_takasiosinnsuisoutei_...` 形式。
  例: `47007_takasiosinnsuisoutei_22itoman.zip`）→「高潮」

ZIPの展開先直下にサブフォルダがある場合（国土数値情報等で「計画規模」「想定最大規模」のようにカテゴリ別に
ファイルが分かれている場合）は、そのサブフォルダ名を **カテゴリ** として扱い、`基本種別名:カテゴリ名` の形で
`hazard_type` に保持します。ただし高潮データは、複数のShapefile（`最大浸水深_糸満市.shp` /
`浸水継続時間_糸満市.shp` 等）が1つのラッパーフォルダにまとめて配布され、フォルダ名だけではレイヤーを
区別できないため、ラッパーフォルダの有無に関わらず、ファイル名から市町村名部分を除いた名前を優先して
カテゴリとします（例: `高潮:最大浸水深` / `高潮:浸水継続時間`）。また、属性に `分類` 列があるレイヤー
（津波浸水想定データ等）は、ポリゴンごとに `分類` の値をカテゴリとして反映します（例:
`津波:0.01m以上0.3m未満`）。土砂災害（A33）は `A33_001`（現象の種類）/`A33_002`（区域区分）の
公式コードリストの名称をそのまま用い、ポリゴンごとに `土砂災害:急傾斜地の崩壊:土砂災害警戒区域(指定済)`
のように反映します（未知のコード値が来た場合も削除せず、コード値自体を保持します）。フォルダ名・
ファイル名・属性の意味は独自解釈せずそのまま使うため、特定の配布元の命名規則以外には依存しません。

座標系はCRS情報がある場合はEPSG:4326へ変換します。CRS情報が無いGeoJSONは既定でWGS84として扱いますが、
CRS情報が無いShapefileについては無条件にEPSG:4326と決めつけず、座標値が経緯度として妥当な範囲
（経度-180〜180・緯度-90〜90）に収まる場合のみEPSG:4326とみなし、収まらない場合はそのレイヤーを座標系
不明として除外します。いずれの形式でもPolygon/MultiPolygon以外の空・不正なジオメトリは除外されます。
`ENABLE_HAZARD_CHECK = False`（既定）の場合はこのセルはスキップされ、ハザードデータなしで距離候補算出のみ
が実行されます。

**注意**: `ENABLE_HAZARD_CHECK = True` にした場合、有効なハザード区域ポリゴンが1件も読み込めなかったときは
「ハザードなし」とみなさず、ここで処理を停止します（判定していないことと、ハザード区域でないことを区別するためです）。


In [ ]:
def _decode_zip_entry_name(member):
    """ZIPエントリのファイル名を復号する。UTF-8フラグ（汎用目的ビットフラグのbit 11）が
    立っていないエントリは、Pythonのzipfileが既定でCP437として解釈するため、CP932
    (Shift-JIS系)の日本語ファイル名を含む配布ZIP（沖縄県公式データ等、UTF-8フラグを立てずに
    作成されたZIP）では文字化けする。その場合は元のバイト列をCP932として再解釈する
    （変換できない場合は元の文字列のまま扱う）。"""
    if member.flag_bits & 0x800:
        return member.filename
    try:
        return member.filename.encode("cp437").decode("cp932")
    except (UnicodeDecodeError, UnicodeEncodeError):
        return member.filename


def _normalize_zip_entry_separators(name):
    """ZIPエントリ名のパス区切りを '/'（ZIP標準の区切り）に正規化する。国土数値情報A31a等、
    一部の公式配布ZIPはディレクトリ区切りに '\\'（バックスラッシュ）を使って格納されているが、
    '\\' はPOSIX環境のPathlibでは区切り文字として扱われずサブフォルダを認識できないため、
    展開前に '/' へ揃える。'/' はWindows・POSIXどちらのPathlibでも区切り文字として扱われるため、
    実行環境（Colab/Linux・Windows）に関わらず同じディレクトリ構造になる。"""
    return name.replace("\\", "/")


def safe_extract_zip(zip_bytes, extract_dir):
    """ZIPをextract_dirへ安全に展開する。各エントリ名は、文字コード復元
    （_decode_zip_entry_name）→パス区切り正規化（_normalize_zip_entry_separators）の順で
    処理してから扱う。絶対パスや'..'を含むなど、正規化後のパスが展開先ディレクトリの外に出る
    エントリが1件でもあれば、展開を一切行わずに例外を送出する（パストラバーサル対策。
    全エントリを先に検査してから展開する）。"""
    extract_dir_abs = Path(extract_dir).resolve()
    with zipfile.ZipFile(io.BytesIO(zip_bytes)) as zip_file:
        resolved_members = []
        for member in zip_file.infolist():
            name = _decode_zip_entry_name(member)
            name = _normalize_zip_entry_separators(name)
            is_dir_entry = name.endswith("/")
            member_path = (extract_dir_abs / name).resolve()
            if member_path != extract_dir_abs and extract_dir_abs not in member_path.parents:
                raise RuntimeError(
                    "ZIP内に不正なパスが含まれているため展開を中止しました"
                    f"（パストラバーサルの可能性）: {name}"
                )
            resolved_members.append((member, member_path, is_dir_entry))

        for member, member_path, is_dir_entry in resolved_members:
            if is_dir_entry:
                member_path.mkdir(parents=True, exist_ok=True)
                continue
            member_path.parent.mkdir(parents=True, exist_ok=True)
            with zip_file.open(member) as source, open(member_path, "wb") as target:
                target.write(source.read())


def find_shapefile_layers(extract_dir):
    """extract_dir配下の.shpを再帰的に探索し、同じディレクトリに.shx/.dbfが揃っているものだけを
    有効なShapefileレイヤーとして返す（.prjが無くても読み込みは試みる。CRSが無いものとして扱う）。
    戻り値は (有効な.shpパスのリスト, .shx/.dbfが揃っていないため除外した件数, 発見した.shp総数)。"""
    shp_paths = sorted(extract_dir.rglob("*.shp"))
    valid_paths = []
    missing_companion_count = 0
    for shp_path in shp_paths:
        siblings = {p.name.lower() for p in shp_path.parent.iterdir()}
        stem_lower = shp_path.stem.lower()
        if f"{stem_lower}.shx" in siblings and f"{stem_lower}.dbf" in siblings:
            valid_paths.append(shp_path)
        else:
            missing_companion_count += 1
    return valid_paths, missing_companion_count, len(shp_paths)


# 国土数値情報A33（土砂災害警戒区域データ）の公式コードリスト。
# A33_001=現象の種類、A33_002=区域区分。名称は公式コードリストのままとし、独自の呼称（イエロー/
# レッド等）へは置き換えない。
A33_PHENOMENON_LABELS = {
    "1": "急傾斜地の崩壊",
    "2": "土石流",
    "3": "地滑り",
}
A33_ZONE_LABELS = {
    "1": "土砂災害警戒区域(指定済)",
    "2": "土砂災害特別警戒区域(指定済)",
    "3": "土砂災害警戒区域(指定前)",
    "4": "土砂災害特別警戒区域(指定前)",
}


def _normalize_code_value(value):
    """GeoJSON属性のコード値を文字列キーへ正規化する。1 / '1' / 1.0 等、読込方法によって型や
    表現が変わり得る値を、整数として解釈できる場合は整数の文字列表現に統一する（表現差で
    コードリストの分類に失敗しないようにするため）。欠損値はNoneを返す。1.5のような整数でない
    値をint(float(value))で丸めてしまうと、既知コードへ誤分類される恐れがあるため丸めず、
    元の値の文字列表現のまま返す（未知のコード値も削除せず識別できる形で保持する）。"""
    if pd.isna(value):
        return None

    text = str(value).strip()

    try:
        number = float(text)
        if number.is_integer():
            return str(int(number))
    except (TypeError, ValueError):
        pass

    return text


def derive_feature_hazard_types(gdf, hazard_type):
    """レイヤーの属性から、行（ポリゴン）ごとのhazard_typeを組み立てる。国土数値情報A33
    （土砂災害警戒区域データ）の 'A33_001'（現象の種類）/'A33_002'（区域区分）属性がある場合は
    公式コードリストの名称をそのまま用いて '<hazard_type>:<現象の種類>:<区域区分>' とする
    （未知のコード値は削除せず、コード値自体をそのまま使う）。津波浸水想定データ等の『分類』属性が
    ある場合は、従来どおり '<hazard_type>:<分類の値>' とする。どちらも無い場合はhazard_typeを
    全行へそのまま適用する。データセットが増えてもこの関数の中だけで判定し、呼び出し側を
    巨大なif文にしない。"""
    if "A33_001" in gdf.columns and "A33_002" in gdf.columns:
        def build_a33_hazard_type(row):
            phenomenon_code = _normalize_code_value(row["A33_001"])
            zone_code = _normalize_code_value(row["A33_002"])
            phenomenon = A33_PHENOMENON_LABELS.get(phenomenon_code, phenomenon_code)
            zone = A33_ZONE_LABELS.get(zone_code, zone_code)
            parts = [part for part in (phenomenon, zone) if part]
            return f"{hazard_type}:{':'.join(parts)}" if parts else hazard_type

        return gdf.apply(build_a33_hazard_type, axis=1)

    if "分類" in gdf.columns:
        return gdf["分類"].apply(
            lambda value: f"{hazard_type}:{value}" if pd.notna(value) and str(value).strip() else hazard_type
        )

    return hazard_type


def load_hazard_layer(source, hazard_type, assume_wgs84_without_crs):
    """1件の空間データ（GeoJSONまたはShapefile）を読み込み、hazard_type/geometryの2列に正規化し、
    EPSG:4326へ統一する。Polygon/MultiPolygon以外、または空・不正なジオメトリは除外する
    （区域判定はPolygon/MultiPolygonのみを対象とするため。LineString等が含まれていても独自に
    面へ変換したりはしない）。

    行ごとのhazard_typeはderive_feature_hazard_typesで組み立てる（『分類』属性やA33の公式属性が
    あれば詳細区分を反映し、無ければhazard_typeをそのまま使う）。

    CRSが取得できる場合はEPSG:4326へ変換する。CRSが取得できない場合、assume_wgs84_without_crsが
    Trueなら（GeoJSON等、既定でWGS84として扱われる形式）そのままEPSG:4326とみなす。Falseの場合
    （Shapefile等）は無条件にEPSG:4326と決めつけず、geometry全体のboundsが経緯度として妥当な範囲
    （経度-180~180・緯度-90~90）に収まる場合のみEPSG:4326と推定し、収まらない場合は座標系を判断
    できないとみなして空のGeoDataFrameを返す（呼び出し側で除外扱いにする）。

    戻り値は (GeoDataFrame, crs_note, non_polygon_count)。crs_noteは
    'assumed_by_bounds' / 'unresolved' / None。non_polygon_countは、有効なジオメトリのうち
    Polygon/MultiPolygon以外（LineString等）だったため区域判定対象外として除外した件数。"""
    gdf = gpd.read_file(source)

    crs_note = None
    if gdf.crs is not None:
        if gdf.crs.to_epsg() != 4326:
            gdf = gdf.to_crs(epsg=4326)
    elif assume_wgs84_without_crs or len(gdf) == 0:
        gdf = gdf.set_crs(epsg=4326)
    else:
        minx, miny, maxx, maxy = gdf.total_bounds
        if -180 <= minx and maxx <= 180 and -90 <= miny and maxy <= 90:
            gdf = gdf.set_crs(epsg=4326)
            crs_note = "assumed_by_bounds"
        else:
            return gpd.GeoDataFrame({"hazard_type": [], "geometry": []}, crs="EPSG:4326"), "unresolved", 0

    valid_mask = gdf.geometry.notna() & gdf.geometry.is_valid
    polygon_mask = gdf.geometry.geom_type.isin(["Polygon", "MultiPolygon"])
    non_polygon_count = int((valid_mask & ~polygon_mask).sum())
    gdf = gdf[valid_mask & polygon_mask]

    row_hazard_types = derive_feature_hazard_types(gdf, hazard_type)

    layer = gpd.GeoDataFrame(
        {"hazard_type": row_hazard_types, "geometry": gdf.geometry.values}, crs="EPSG:4326"
    )
    return layer, crs_note, non_polygon_count


def _takashio_category_from_stem(stem):
    """高潮データ（例: 最大浸水深_糸満市.shp）のファイル名から、末尾の市町村名部分を除いた
    カテゴリ名を返す（例: '最大浸水深_糸満市' → '最大浸水深'）。区切りが無ければそのまま返す。"""
    prefix, sep, _rest = stem.rpartition("_")
    return prefix if sep else stem


def resolve_layer_hazard_type(base_hazard_type, vector_path, extract_dir):
    """ハザードのカテゴリをhazard_typeへ反映する。国土数値情報等では洪水の中でも「計画規模」
    「想定最大規模」等がZIP展開先直下のサブフォルダで分かれて配布されるため、そのサブフォルダ名を
    そのままカテゴリ名として使う（フォルダ名の意味は独自解釈せず、特定の配布元のディレクトリ名には
    依存しない）。ただし高潮（沖縄県高潮浸水想定データ）は複数のShapefileレイヤーが1つのラッパー
    フォルダにまとめて配布され、フォルダ名だけではレイヤーを区別できないため、フォルダの有無に
    関わらずファイル名から市町村名部分を除いたカテゴリ（例: 最大浸水深/浸水継続時間）を優先して
    用いる。サブフォルダも高潮の命名規則も無い場合はカテゴリなし（基本種別名のみ）とする。"""
    if base_hazard_type == "高潮":
        category = _takashio_category_from_stem(vector_path.stem)
    else:
        rel_parts = vector_path.relative_to(extract_dir).parts
        category = rel_parts[0] if len(rel_parts) > 1 else None
    return f"{base_hazard_type}:{category}" if category else base_hazard_type


def _detect_a31a_hazard_type(filename):
    """国土数値情報A31a（洪水浸水想定区域データ）のファイル名規則
    （例: A31a-25_47_10_GEOJSON.zip）から、年度・都道府県コードには依存せず、
    河川区分（10/20）のみで洪水の種別名を判定する。一致しなければNoneを返す。"""
    match = re.match(r"^A31a-\d+_\d+_(10|20)(?=[_.])", filename)
    if not match:
        return None
    river_classification_labels = {
        "10": "洪水（洪水予報河川・水位周知河川）",
        "20": "洪水（その他の河川）",
    }
    return river_classification_labels[match.group(1)]


def _detect_a33_hazard_type(filename):
    """国土数値情報A33（土砂災害警戒区域データ）のファイル名規則
    （例: A33-25_47_GEOJSON.zip）から、年度・都道府県コードには依存せず土砂災害と判定する。
    ファイル名の先頭部分だけを見るため、Colab等がファイル名重複を避けて末尾に付与する
    '(1)' 等の連番があっても判定できる。一致しなければNoneを返す。"""
    if re.match(r"^A33-\d+_\d+_GEOJSON", filename):
        return "土砂災害"
    return None


def _detect_tsunami_level_hazard_type(filename):
    """沖縄県津波浸水想定データのファイル名規則（例: level1_1cm-30cm.zip、level1～level7）から
    津波と判定する。浸水深の分類はファイル名のlevel番号からは推測せず、実データのShapefile内
    『分類』属性（load_hazard_layerで反映）を優先する。"""
    if re.match(r"^level[1-7](?=[_.])", filename, re.IGNORECASE):
        return "津波"
    return None


def _detect_takashio_hazard_type(filename):
    """沖縄県高潮浸水想定データのファイル名規則
    （例: 47007_takasiosinnsuisoutei_22itoman.zip）から高潮と判定する。"""
    if re.match(r"^\d+_takasiosinnsuisoutei_", filename):
        return "高潮"
    return None


# 既知の公式データ配布ファイル名パターンの判定関数一覧。今後、新しい公式データにも対応する場合は
# ここに判定関数を追加すればよく、巨大なif文へは積み上げない（現時点ではA31a・A33・
# 津波(level1~7)・高潮(takasiosinnsuisoutei)のみ、実データで確認できたファイル名規則として
# 実装している）。
KNOWN_HAZARD_FILENAME_DETECTORS = [
    _detect_a31a_hazard_type,
    _detect_a33_hazard_type,
    _detect_tsunami_level_hazard_type,
    _detect_takashio_hazard_type,
]


def detect_hazard_type(filename):
    """既知の公式データ配布ファイル名パターンからハザード種別名を自動判定する。ZIP・GeoJSON単体の
    どちらのファイル名にも同じ規則を適用する（形式ごとに判定ロジックを二重実装しない）。曖昧な推測は
    せず、既知のパターンのいずれにも一致しない場合はNoneを返す（呼び出し側で利用者入力へフォールバック
    する）。"""
    for detector in KNOWN_HAZARD_FILENAME_DETECTORS:
        hazard_type = detector(filename)
        if hazard_type is not None:
            return hazard_type
    return None


def load_hazard_upload(filename, file_bytes, hazard_type):
    """1つのアップロード（.geojson または国・県等の公式配布ZIP）から、GeoDataFrameを組み立てる。
    ZIPの場合は安全に展開し、内部のディレクトリ構造に関わらず配下の.geojsonと.shp（Shapefile。
    同名の.shx/.dbfが揃っているものに限る）を再帰的に探索する（特定の配布元のファイル名・
    ディレクトリ名には依存しない）。サブフォルダがあればカテゴリとしてhazard_typeに反映し、
    サブフォルダがなければ渡された基本種別名をそのままhazard_typeとする（高潮のみファイル名から
    カテゴリを補う。resolve_layer_hazard_type参照）。利用者には集約したサマリのみ表示し、
    ファイルごとの詳細ログは出さない。"""
    suffix = Path(filename).suffix.lower()
    if suffix not in (".geojson", ".zip"):
        raise ValueError(f"'{filename}' は対応していない形式です。.geojson または .zip を選択してください。")

    layers = []
    empty_file_count = 0
    unresolved_crs_count = 0
    assumed_by_bounds_count = 0
    missing_shapefile_companion_count = 0
    excluded_geometry_type_count = 0

    if suffix == ".geojson":
        layer, crs_note, non_polygon_count = load_hazard_layer(
            io.BytesIO(file_bytes), hazard_type, assume_wgs84_without_crs=True
        )
        excluded_geometry_type_count += non_polygon_count
        if len(layer) == 0:
            empty_file_count += 1
        else:
            layers.append(layer)
    else:
        with tempfile.TemporaryDirectory(prefix="hazard_zip_") as extract_dir_str:
            extract_dir = Path(extract_dir_str)
            safe_extract_zip(file_bytes, extract_dir)
            print(f"'{filename}' を展開しました。")

            geojson_paths = sorted(extract_dir.rglob("*.geojson"))
            shp_paths, missing_shapefile_companion_count, total_shp_found = find_shapefile_layers(extract_dir)

            if not geojson_paths and total_shp_found == 0:
                raise RuntimeError(
                    f"'{filename}' 内に.geojsonまたはShapefile(.shp)が見つかりませんでした。"
                )

            print(f"Shapefileを {len(shp_paths)}レイヤー検出しました。")
            print(f"GeoJSONを {len(geojson_paths)}ファイル検出しました。")

            all_vector_paths = [(p, True) for p in geojson_paths] + [(p, False) for p in shp_paths]

            categories = sorted(
                {
                    p.relative_to(extract_dir).parts[0]
                    for p, _ in all_vector_paths
                    if len(p.relative_to(extract_dir).parts) > 1
                }
            )
            if categories:
                print(f"{len(categories)}カテゴリ（サブフォルダ）を検出しました。")

            for vector_path, is_geojson in all_vector_paths:
                file_hazard_type = resolve_layer_hazard_type(hazard_type, vector_path, extract_dir)
                layer, crs_note, non_polygon_count = load_hazard_layer(
                    vector_path, file_hazard_type, assume_wgs84_without_crs=is_geojson
                )
                excluded_geometry_type_count += non_polygon_count
                if crs_note == "unresolved":
                    unresolved_crs_count += 1
                    continue
                if crs_note == "assumed_by_bounds":
                    assumed_by_bounds_count += 1
                if len(layer) == 0:
                    empty_file_count += 1
                else:
                    layers.append(layer)

    if missing_shapefile_companion_count:
        print(
            f"{missing_shapefile_companion_count}件のShapefileは.shx/.dbfが揃っていないため"
            "読み込めませんでした。"
        )
    if assumed_by_bounds_count:
        print(
            f"{assumed_by_bounds_count}件はCRS情報が無いため、座標値から経緯度データと判断して"
            "EPSG:4326として読み込みました。"
        )
    if excluded_geometry_type_count:
        print(f"{excluded_geometry_type_count}件は区域判定対象外のgeometry typeのため除外しました。")
    if empty_file_count:
        print(f"{empty_file_count}ファイルは有効なPolygon/MultiPolygonを含まないため除外しました。")
    if unresolved_crs_count:
        print(
            f"{unresolved_crs_count}件は座標系を特定できないため除外しました"
            "（CRS情報が無く、座標値も経緯度として妥当な範囲ではありませんでした）。"
        )

    if not layers:
        raise RuntimeError(
            f"'{filename}' から有効なハザード区域(Polygon/MultiPolygon)を1件も読み込めませんでした。"
        )

    combined = gpd.GeoDataFrame(pd.concat(layers, ignore_index=True), crs="EPSG:4326")
    print(f"有効なハザードポリゴンを {len(combined)}件読み込みました。")
    combined_hazard_types = sorted(combined["hazard_type"].unique())
    if len(combined_hazard_types) == 1:
        print(f"hazard_type='{combined_hazard_types[0]}'")
    else:
        print(f"hazard_type: {', '.join(combined_hazard_types)}")
    return combined


hazard_gdf = None

if ENABLE_HAZARD_CHECK:
    print("ハザード区域のデータ（.geojson または国・県等の公式配布ZIP）をアップロードしてください（複数選択可）。")
    uploaded_hazards = files.upload()

    _stage_start = time.perf_counter()

    hazard_layers = []
    for hazard_filename, hazard_bytes in uploaded_hazards.items():
        detected_hazard_type = detect_hazard_type(hazard_filename)
        if detected_hazard_type is not None:
            hazard_type = detected_hazard_type
            print(f"'{hazard_filename}'")
            print(f"→ ハザード種別を自動判定しました: {hazard_type}")
        else:
            hazard_type = input(
                f"'{hazard_filename}' のハザード種別を自動判定できませんでした。\n"
                "ハザード種別名を入力してください（例: 洪水, 土砂災害, 津波, 高潮。"
                "ZIP内にサブフォルダがあれば、フォルダ名がカテゴリとして自動的に追加されます）: "
            ).strip()
            if not hazard_type:
                hazard_type = hazard_filename
        # 複数ファイルをまとめてアップロードした場合、1ファイルの読込に失敗しても
        # （形式非対応・ZIP内にgeojson/shpが無い・有効なポリゴンが無い等）、他の正常な
        # ファイルの処理を止めない（一括アップロードのうち一部だけ不正でも、有効な分は使えるようにする）。
        try:
            layer = load_hazard_upload(hazard_filename, hazard_bytes, hazard_type)
        except (RuntimeError, ValueError) as error:
            print(f"'{hazard_filename}' の読込をスキップしました（詳細: {error}）。")
            continue
        hazard_layers.append(layer)

    if hazard_layers:
        hazard_gdf = gpd.GeoDataFrame(pd.concat(hazard_layers, ignore_index=True), crs="EPSG:4326")

    if hazard_gdf is None or len(hazard_gdf) == 0:
        raise RuntimeError(
            "ENABLE_HAZARD_CHECK=True ですが、有効なハザード区域(Polygon/MultiPolygon)を1件も読み込めませんでした。"
            "GeoJSON/ZIPファイルが正しくアップロードされているか、ジオメトリ形式を確認してください。"
            "ハザード判定を行わない場合は、上の「利用者設定」で ENABLE_HAZARD_CHECK=False にしてください。"
        )

    print(f"ハザードデータを合計 {len(hazard_gdf)}件読み込みました。")

    TIMINGS["hazard"] = time.perf_counter() - _stage_start
    print(f"[処理時間] ハザードデータ読込・統合: {TIMINGS['hazard']:.2f}秒")
else:
    print("ENABLE_HAZARD_CHECK=False のため、ハザードデータの読込をスキップします。")
    TIMINGS["hazard"] = 0.0


In [ ]:
# ===== 距離計算・ハザード判定関数 =====
# 各要支援者について、有効な避難所すべてとの直線距離（geopy.distance.geodesic、メートル単位）を計算し、
# 近い順に TOP_N 件を候補として算出します（道路距離ではありません）。並び替えは丸める前の距離で行い、
# メートル単位への丸め（小数1桁）はCSVに出力する値を作成する際にのみ行います。距離が同一の場合でも
# 結果順が実行ごとにばらつかないよう、避難所名を用いて順序を安定させます。
#
# ハザード判定関数は、要支援者地点・候補避難所地点がハザード区域の内部または境界上にあるか、また
# 両地点を結ぶ直線がハザード区域と交差するかを判定します（直線交差は道路上の避難経路判定ではありません）。
# 座標が欠損・範囲外で判定できない場合は、区域外(False)と混同しないよう NaN を返します。


def compute_candidates(resident_lat, resident_lon, shelters_df, top_n):
    """要支援者の座標から近い順に避難所候補を [(名前, 距離m, 緯度, 経度, 災害種別対応区分), ...] で返す。
    座標が欠損、または緯度・経度が有効範囲外であればNoneを返す（geodesic()に不正値を渡さない）。
    距離は丸めずに返す（並び替え後、出力時にのみ丸める）。対応区分は災害種別列が無ければNoneのまま。"""
    if not is_valid_coordinate(resident_lat, resident_lon):
        return None

    has_disaster_info = "_disaster_support" in shelters_df.columns
    resident_coord = (resident_lat, resident_lon)
    records = []
    for _, shelter in shelters_df.iterrows():
        shelter_coord = (shelter["latitude"], shelter["longitude"])
        distance_m = geodesic(resident_coord, shelter_coord).meters
        disaster_support = shelter["_disaster_support"] if has_disaster_info else np.nan
        records.append(
            (shelter["name"], distance_m, shelter["latitude"], shelter["longitude"], disaster_support)
        )

    records.sort(key=lambda record: (record[1], record[0]))
    return records[:top_n]


def hazard_types_at_point(lat, lon, hazard_area):
    """座標がハザード区域の内部または境界上にあるかどうかと、該当するhazard_type（;区切り）を返す。
    座標が欠損・範囲外の場合は判定不能としてnp.nanを返す（区域外=Falseと混同しない）。"""
    if hazard_area is None or len(hazard_area) == 0:
        return False, ""
    if not is_valid_coordinate(lat, lon):
        return np.nan, np.nan

    point = Point(lon, lat)
    hit_types = sorted(hazard_area.loc[hazard_area.geometry.intersects(point), "hazard_type"].unique())
    return (len(hit_types) > 0), ";".join(hit_types)


def hazard_types_on_line(lat1, lon1, lat2, lon2, hazard_area):
    """2地点を結ぶ直線がハザード区域と交差するかどうかと、該当するhazard_typeを返す（道路経路上の判定ではない）。
    いずれかの座標が欠損・範囲外の場合は判定不能としてnp.nanを返す。"""
    if hazard_area is None or len(hazard_area) == 0:
        return False, ""
    if not is_valid_coordinate(lat1, lon1) or not is_valid_coordinate(lat2, lon2):
        return np.nan, np.nan

    line = LineString([(lon1, lat1), (lon2, lat2)])
    hit_types = sorted(hazard_area.loc[hazard_area.geometry.intersects(line), "hazard_type"].unique())
    return (len(hit_types) > 0), ";".join(hit_types)


print("距離計算・ハザード判定関数を定義しました。")


In [ ]:
# ===== 候補算出とハザード判定の実行 =====
# 要支援者ごとに、避難所候補・距離・（ENABLE_HAZARD_CHECK=True の場合のみ）ハザード判定結果を組み立てます。
# 距離による候補順位はハザード判定結果によって変更されません。座標がない・不正な要支援者の行も削除せず、
# 候補・距離を空欄のまま保持します（match_status 列で ok / no_coordinates(座標欄が空欄) /
# invalid_coordinates(値はあるが範囲外) の状態を確認できます）。
# ハザード関連列は、有効な座標で実際に判定できた場合のみ True/False とし、要支援者座標が欠損・不正な場合や
# 候補自体が存在しない場合は NaN（判定不能）のまま出力します（候補避難所地点は常に有効座標を持つため、
# 候補が存在すれば通常どおり True/False で判定します）。
# 避難所一覧に災害種別列があった場合は candidate_N_disaster_support 列に、災害種別ごとの対応区分
# （例: 洪水:対応済み;高潮:2階以上であれば対応済み）を出力します（対応区分によって候補の順位変更・自動除外は行いません）。

_stage_start = time.perf_counter()

result_records = []

for _, resident in residents.iterrows():
    lat, lon = resident["latitude"], resident["longitude"]
    candidates = compute_candidates(lat, lon, shelters_valid, TOP_N)

    if pd.isna(lat) or pd.isna(lon):
        coord_status = "no_coordinates"
    elif not is_valid_coordinate(lat, lon):
        coord_status = "invalid_coordinates"
    else:
        coord_status = "ok"

    record = {}

    if ENABLE_HAZARD_CHECK:
        in_hazard, hazard_types = hazard_types_at_point(lat, lon, hazard_gdf)
        record["resident_in_hazard"] = in_hazard
        record["resident_hazard_types"] = hazard_types

    for i in range(TOP_N):
        n = i + 1
        candidate_col = f"candidate_{n}"
        distance_col = f"distance_{n}_m"
        disaster_col = f"candidate_{n}_disaster_support"
        shelter_hazard_col = f"candidate_{n}_shelter_in_hazard"
        shelter_hazard_types_col = f"candidate_{n}_shelter_hazard_types"
        line_hazard_col = f"candidate_{n}_straight_line_intersects_hazard"
        line_hazard_types_col = f"candidate_{n}_straight_line_hazard_types"

        if candidates is not None and i < len(candidates):
            shelter_name, distance_m, shelter_lat, shelter_lon, disaster_support = candidates[i]
            record[candidate_col] = shelter_name
            record[distance_col] = round(distance_m, 1)

            if HAS_DISASTER_TYPE_COLUMNS:
                record[disaster_col] = disaster_support

            if ENABLE_HAZARD_CHECK:
                shelter_in_hazard, shelter_hazard_types = hazard_types_at_point(
                    shelter_lat, shelter_lon, hazard_gdf
                )
                record[shelter_hazard_col] = shelter_in_hazard
                record[shelter_hazard_types_col] = shelter_hazard_types

                line_intersects, line_hazard_types = hazard_types_on_line(
                    lat, lon, shelter_lat, shelter_lon, hazard_gdf
                )
                record[line_hazard_col] = line_intersects
                record[line_hazard_types_col] = line_hazard_types
        else:
            record[candidate_col] = np.nan
            record[distance_col] = np.nan
            if HAS_DISASTER_TYPE_COLUMNS:
                record[disaster_col] = np.nan
            if ENABLE_HAZARD_CHECK:
                record[shelter_hazard_col] = np.nan
                record[shelter_hazard_types_col] = np.nan
                record[line_hazard_col] = np.nan
                record[line_hazard_types_col] = np.nan

    record["match_status"] = coord_status
    result_records.append(record)

results_df = pd.DataFrame(result_records)
final_df = pd.concat([residents.reset_index(drop=True), results_df], axis=1)

# match_statusは「座標→避難所候補の突合に成功したか」を示す列で、候補・ハザード関連の全列より
# 前（共通住民CSVの5列の直後）に置いた方が、職員が1行を横方向に追ったときに読みやすいため並べ替える
# （値の計算方法は変更せず、列の並び順だけを変える）。
base_columns = [c for c in ("resident_id", "address", "latitude", "longitude", "geocode_status") if c in final_df.columns]
other_columns = [c for c in final_df.columns if c not in base_columns and c != "match_status"]
final_df = final_df[base_columns + ["match_status"] + other_columns]

TIMINGS["candidates"] = time.perf_counter() - _stage_start
print("候補算出が完了しました。")
print(f"[処理時間] 避難所候補算出＋ハザード判定: {TIMINGS['candidates']:.2f}秒")


In [ ]:
# ===== 結果確認・CSV出力 =====
# CSVを出力する前に、Notebook上で処理結果の概要と先頭数行を確認します。
# 結果は Excelで文字化けしにくい utf-8-sig（UTF-8 BOM付き）でCSVに出力し、ブラウザへダウンロードします。
# 出力CSVには個人情報が含まれ得るため、取り扱いに注意してください。

_stage_start = time.perf_counter()

total_residents = len(final_df)
ok_count = int((final_df["match_status"] == "ok").sum())
no_coord_count = int((final_df["match_status"] == "no_coordinates").sum())
invalid_coord_count = int((final_df["match_status"] == "invalid_coordinates").sum())

print(f"要支援者件数: {total_residents}件")
print(f"距離計算できた件数: {ok_count}件")
print(f"座標が空欄のため距離計算できなかった件数: {no_coord_count}件")
print(f"座標が範囲外で不正なため距離計算できなかった件数: {invalid_coord_count}件")
print(f"距離計算に使用した有効な避難所件数: {len(shelters_valid)}件")

if ENABLE_HAZARD_CHECK:
    resident_hazard_count = int((final_df["resident_in_hazard"] == True).sum())
    print(f"ハザード区域内（境界上含む）にいる要支援者数: {resident_hazard_count}件")

    shelter_hazard_flags = [
        final_df[f"candidate_{i + 1}_shelter_in_hazard"] == True for i in range(TOP_N)
    ]
    line_hazard_flags = [
        final_df[f"candidate_{i + 1}_straight_line_intersects_hazard"] == True for i in range(TOP_N)
    ]
    shelter_hazard_count = int(pd.concat(shelter_hazard_flags, axis=1).sum().sum())
    line_hazard_count = int(pd.concat(line_hazard_flags, axis=1).sum().sum())

    print(f"ハザード区域内にある候補避難所の件数（延べ、TOP_N分の合計）: {shelter_hazard_count}件")
    print(f"候補避難所への直線がハザード区域と交差する件数（延べ、TOP_N分の合計）: {line_hazard_count}件")

display(final_df.head())

OUTPUT_FILENAME = "assigned_shelters.csv"

final_df.to_csv(OUTPUT_FILENAME, index=False, encoding="utf-8-sig")
TIMINGS["output"] = time.perf_counter() - _stage_start
print(f"'{OUTPUT_FILENAME}' を出力しました。")
print(f"[処理時間] 結果CSV生成: {TIMINGS['output']:.2f}秒")

total_time = sum(TIMINGS.values())
print("\n処理時間:")
print(f"  要支援者CSV読込        {TIMINGS.get('residents_csv', 0):.1f}秒")
print(f"  避難所データ準備        {TIMINGS.get('shelters', 0):.1f}秒")
print(f"  ハザードデータ読込      {TIMINGS.get('hazard', 0):.1f}秒")
print(f"  候補・ハザード判定      {TIMINGS.get('candidates', 0):.1f}秒")
print(f"  CSV出力                {TIMINGS.get('output', 0):.1f}秒")
print(f"  合計                   {total_time:.1f}秒")

files.download(OUTPUT_FILENAME)
